# 05 - SASRec (Self-Attentive Sequential Recommendation)

SASRec uses **causal self-attention** over the city sequence: at each step it learns which earlier stops matter most for predicting the next, rather than compressing everything into one state. It is the natural place to add side features.

All logic is in `src/models/sasrec.py` (a hand-rolled transformer so the causal + padding masks are fully controlled). This notebook runs the required **side-feature ablation** and logs results through the shared `evaluate.py`.

> **Compute note:** on a small CPU subsample SASRec can trail FPMC, because short trips are dominated by the last-city transition and transformers are data-hungry. Its advantage is expected on the **full dataset with a GPU** (Colab: Runtime -> Change runtime type -> GPU, then pass `device='cuda'`).

In [ ]:
import os, sys
def find_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'src', 'preprocessing.py')):
            return d
        d = os.path.dirname(d)
    return os.path.abspath(os.path.join(os.getcwd(), '..'))
ROOT = find_root(); sys.path.insert(0, ROOT)
import torch
from src.preprocessing import load_processed, preprocess
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
try:
    data = load_processed(os.path.join(ROOT, 'data', 'processed'))
except FileNotFoundError:
    data = preprocess(data_dir=os.path.join(ROOT, 'data'), sample_n_trips=5000)
print('loaded:', data['meta'])

## Ablation - side features on/off, and how they combine

We compare (a) no side features, (b) side features **summed** into the city embedding, and (c) side features **concatenated** then projected back to the model width. The features are trip-level context: `device_class`, `booker_country`, `affiliate_id`. This isolates which context actually moves the metrics and whether sum or concat is better.

In [ ]:
from src.models.sasrec import run_sasrec, SIDE_FEATURES
from src.evaluate import append_result
RESULTS = os.path.join(ROOT, 'results.csv')

configs = [
    ('SASRec-no-side',      (),            'none'),
    ('SASRec-side-sum',     SIDE_FEATURES, 'sum'),
    ('SASRec-side-concat',  SIDE_FEATURES, 'concat'),
]
for name, feats, mode in configs:
    print(f'--- {name} ---')
    metrics, _, val = run_sasrec(data, side_feats=feats, side_mode=mode,
                                 dim=64, epochs=40, device=DEVICE, verbose=True)
    print('TEST:', {k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()})
    append_result(RESULTS, name, metrics)

## Per-feature ablation (optional, for the report)

To identify *which single* side feature helps, run one feature at a time:

```python
for f in SIDE_FEATURES:
    m,_,_ = run_sasrec(data, side_feats=(f,), side_mode='sum', device=DEVICE, verbose=False)
    print(f, m['NDCG@10'])
```

## What to write up

- Report the side-feature ablation (none vs sum vs concat, and per-feature).
- Contrast SASRec with FPMC/Markov by **trip length** (the comparison notebook slices this): the attention advantage should show up on longer trips.
- Be explicit about the compute story: subsample+CPU vs full+GPU, and how the numbers move.